# Create EFSD Awards (European Foundation for the Study of Diabetes)

Creates EFSD awards from the foundation's own recipients pages,
https://www.europeandiabetesfoundation.org/recipients/ — one static HTML page per
grant/fellowship programme listing, per award year, each recipient's name,
institution - city/country and project title (mentorship programme: the mentor).

**422 awards, 18 programmes, 2005-2026.** Most programmes are co-funded with a partner
(Lilly, Novo Nordisk / Novo Nordisk Foundation, Boehringer Ingelheim, MSD, Sanofi,
AstraZeneca, JDRF, Japan Diabetes Society, EUDF). EFSD runs the call, review and award, so
every award ships under EFSD; `funder_scheme` is the programme name as EFSD publishes it,
which names the partner (e.g. "EFSD/Lilly Young Investigator Research Award Programme",
"EFSD Rising Star Fellowship Programme supported by Novo Nordisk"). The raw table also has
a `partner` column.

**Scope:** every programme on the recipients page, including the training ones (Rising Star
fellowships, Future Leaders Mentorship for Clinical Diabetologists, JDS reciprocal travel
fellowships). Nothing filtered. Mentorship awards have no project title; `display_name` is
"<programme> — mentor: <mentor>".

**Not published by EFSD:** amounts (§6.7 amount check waived), start/end dates (only the
award year: `start_date` = 1 January of that year) and EFSD's internal grant numbers.
`funder_award_id` is therefore the synthetic `EFSD-<programme>-<year>-<recipient>` key
(§2.1.1 step 2); citing works write free-text forms ("EFSD/Lilly 2015", SmartSimple-style
numbers), so no existing citation stubs collapse.

**Not the same funder as EASD** (European Association for the Study of Diabetes,
F4320324757), EFSD's parent association. EFSD is F4320320870.

**Prerequisites:** run `scripts/local/efsd_to_s3.py` first. It uploads
`s3://openalex-ingest/awards/efsd/efsd_projects.parquet` (§1.4 shrink guard).

**Funder (Path A, F4320* Crossref-registered):** funder_id `4320320870`; display_name,
ror (05tgz4m05) and doi (10.13039/501100001648) read from `openalex.funders.funders`.

**Priority:** `545` (direct funder ingest; higher wins under the 2026-06-20 DESC dedup).

## Step 1: Create Staging Table from S3

In [ ]:
%sql
CREATE OR REPLACE TABLE openalex.awards.efsd_raw
USING delta
AS
SELECT *, current_timestamp() as databricks_ingested_at
FROM parquet.`s3a://openalex-ingest/awards/efsd/efsd_projects.parquet`;

In [ ]:
%sql
SELECT programme, partner, funding_type, COUNT(*) as n, MIN(award_year) as first_year, MAX(award_year) as last_year
FROM openalex.awards.efsd_raw
GROUP BY 1, 2, 3 ORDER BY n DESC;

In [ ]:
%sql
DESCRIBE openalex.awards.efsd_raw;

In [ ]:
%sql
SELECT * FROM openalex.awards.efsd_raw LIMIT 5;

## Step 1.6: Funder existence check (Path A)
F4320320870 is a Crossref-registered funder, so it MUST resolve to exactly 1 row in
`openalex.funders.funders`. If 0 rows, STOP and flag.

In [ ]:
%sql
SELECT funder_id, display_name, ror_id, doi, country_code
FROM openalex.funders.funders
WHERE funder_id = 4320320870;

## Step 2: Create EFSD Awards Table

In [ ]:
%sql
CREATE OR REPLACE TABLE openalex.awards.efsd_awards
USING delta
AS
WITH
src_funder AS (
    SELECT funder_id, display_name, ror_id, doi
    FROM openalex.funders.funders
    WHERE funder_id = 4320320870  -- European Foundation for the Study of Diabetes
),

parsed AS (
    SELECT
        g.*,
        -- struct field order MUST match openalex_awards_raw: ...orcid, role_start, affiliation.
        CASE WHEN NULLIF(TRIM(g.family_name), '') IS NOT NULL THEN named_struct(
            'given_name', NULLIF(TRIM(g.given_name), ''),
            'family_name', NULLIF(TRIM(g.family_name), ''),
            'orcid', CAST(NULL AS STRING),
            'role_start', CAST(NULL AS DATE),
            'affiliation', named_struct(
                'name', NULLIF(TRIM(g.institution), ''),
                'country', NULLIF(TRIM(g.country), ''),
                'ids', CAST(NULL AS ARRAY<STRUCT<id:STRING, type:STRING, asserted_by:STRING>>)
            )
        ) END AS pi
    FROM openalex.awards.efsd_raw g
)

SELECT
    abs(xxhash64(CONCAT(f.funder_id, ':', LOWER(TRIM(p.funder_award_id))))) % 9000000000 as id,
    COALESCE(NULLIF(TRIM(p.title), ''),
             CASE WHEN p.mentor IS NOT NULL THEN CONCAT(p.programme, ' — mentor: ', TRIM(p.mentor)) END,
             p.programme) as display_name,
    CAST(NULL AS STRING) as description,
    f.funder_id,
    TRIM(p.funder_award_id) as funder_award_id,
    CAST(NULL AS DOUBLE) as amount,
    CAST(NULL AS STRING) as currency,
    struct(
        CONCAT('https://openalex.org/F', f.funder_id) as id,
        f.display_name,
        f.ror_id,
        f.doi
    ) as funder,
    p.funding_type as funding_type,
    p.programme as funder_scheme,
    'efsd_recipients' as provenance,
    TRY_TO_DATE(CONCAT(p.award_year, '-01-01'), 'yyyy-MM-dd') as start_date,
    CAST(NULL AS DATE) as end_date,
    TRY_CAST(p.award_year AS INT) as start_year,
    CAST(NULL AS INT) as end_year,
    p.pi as lead_investigator,
    CAST(NULL AS STRUCT<given_name:STRING, family_name:STRING, orcid:STRING, role_start:DATE,
         affiliation:STRUCT<name:STRING, country:STRING, ids:ARRAY<STRUCT<id:STRING, type:STRING, asserted_by:STRING>>>>) as co_lead_investigator,
    CASE WHEN p.pi IS NOT NULL THEN array(p.pi) END as investigators,
    p.landing_page_url as landing_page_url,
    CAST(NULL AS STRING) as doi,
    concat('https://api.openalex.org/works?filter=awards.id:G',
           abs(xxhash64(CONCAT(f.funder_id, ':', LOWER(TRIM(p.funder_award_id))))) % 9000000000) as works_api_url,
    current_timestamp() as created_date,
    current_timestamp() as updated_date
FROM parsed p
CROSS JOIN src_funder f
WHERE p.funder_award_id IS NOT NULL AND TRIM(p.funder_award_id) != '';

In [ ]:
%sql
-- §2.1.1 shape check: synthetic EFSD-<programme>-<year>-<recipient>; expect bad_shape = 0, duplicate_ids = 0.
SELECT
    COUNT(*) as total,
    SUM(CASE WHEN funder_award_id RLIKE '^EFSD-[a-z0-9-]+-(19|20)[0-9]{2}-[a-z0-9-]+$' THEN 0 ELSE 1 END) as bad_shape,
    COUNT(*) - COUNT(DISTINCT id) as duplicate_ids
FROM openalex.awards.efsd_awards;

In [ ]:
%sql
-- Remove previous data for this source before inserting fresh data.
DELETE FROM openalex.awards.openalex_awards_raw
WHERE provenance = 'efsd_recipients' AND priority = 545;

-- Insert into openalex_awards_raw with priority.
-- Priority 545: direct-from-funder ingest of EFSD's own recipients pages.
-- Higher wins under the 2026-06-20 DESC dedup (oxjob #500).
INSERT INTO openalex.awards.openalex_awards_raw
SELECT
    id, display_name, description, funder_id, funder_award_id, amount, currency,
    funder, funding_type, funder_scheme, provenance, start_date, end_date,
    start_year, end_year, lead_investigator, co_lead_investigator, investigators,
    landing_page_url, doi, works_api_url, created_date, updated_date,
    545 as priority
FROM openalex.awards.efsd_awards;

## Verification Queries

In [ ]:
%sql
SELECT COUNT(*) as total_awards, COUNT(DISTINCT id) as distinct_ids
FROM openalex.awards.efsd_awards;

In [ ]:
%sql
SELECT id, display_name, funder_award_id, funder_scheme, funding_type, start_year,
       lead_investigator.given_name, lead_investigator.family_name,
       lead_investigator.affiliation.name, lead_investigator.affiliation.country
FROM openalex.awards.efsd_awards LIMIT 20;

In [ ]:
%sql
SELECT funder_scheme, funding_type, COUNT(*) as cnt
FROM openalex.awards.efsd_awards
GROUP BY 1, 2 ORDER BY cnt DESC;

In [ ]:
%sql
SELECT start_year, COUNT(*) as cnt
FROM openalex.awards.efsd_awards
WHERE start_year IS NOT NULL GROUP BY start_year ORDER BY start_year;

In [ ]:
%sql
-- §6.4a frequency check: PI top-20 must be a real long-tail.
SELECT lead_investigator.given_name AS given, lead_investigator.family_name AS family, COUNT(*) AS n
FROM openalex.awards.efsd_awards
GROUP BY 1, 2 ORDER BY n DESC LIMIT 20;

In [ ]:
%sql
-- §6.4a title frequency check.
SELECT display_name, COUNT(*) AS n
FROM openalex.awards.efsd_awards
GROUP BY 1 ORDER BY n DESC LIMIT 10;

In [ ]:
%sql
-- §6.3 coverage (expect 100% title/PI/start_date; amount 0% because EFSD publishes none).
SELECT
    COUNT(*) as total,
    COUNT(display_name) as has_title,
    COUNT(amount) as has_amount,
    COUNT(start_date) as has_start_date,
    COUNT(lead_investigator.family_name) as has_pi_name,
    COUNT(lead_investigator.affiliation.name) as has_institution,
    COUNT(lead_investigator.affiliation.country) as has_country
FROM openalex.awards.efsd_awards;

In [ ]:
%sql
-- Confirm rows reached the shared raw table at the assigned priority (§6.8).
SELECT provenance, priority, COUNT(*) as n
FROM openalex.awards.openalex_awards_raw
WHERE provenance = 'efsd_recipients'
GROUP BY provenance, priority;